In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-da537b48-3270-41fe-9b69-07cc0176f64e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (93ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (1619ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-op

In [5]:
# 2. Carregamento dos Dados
products_df = spark.read.option('delimiter', ',') \
              .option('header', 'true') \
              .option('nullValue', 'NULL') \
              .csv('s3a://last-mile-optimization-raw/dataset-orders/olist_products_dataset.csv')

translation_df = spark.read.option('delimiter', ',') \
              .option('header', 'true') \
              .option('nullValue', 'NULL') \
              .csv('s3a://last-mile-optimization-raw/dataset-orders/product_category_name_translation.csv')

# products_df = spark.read.csv("olist_products_dataset.csv", header=True, inferSchema=True)
# translation_df = spark.read.csv("product_category_name_translation.csv", header=True, inferSchema=True)

print("Schema do DataFrame de Produtos:")
products_df.printSchema()

print("\nSchema do DataFrame de Tradução:")
translation_df.printSchema()

Schema do DataFrame de Produtos:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)


Schema do DataFrame de Tradução:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [7]:
# 3. Remoção de Colunas
columns_to_drop = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]
products_df = products_df.drop(*columns_to_drop)

print("DataFrame de Produtos após remoção de colunas:")
products_df.printSchema()

DataFrame de Produtos após remoção de colunas:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [8]:
# 4. Tradução de Categorias
# Juntando os dataframes para obter os nomes em inglês
products_translated_df = products_df.join(
    translation_df,
    on="product_category_name",
    how="left"
).withColumnRenamed(
    "product_category_name_english", "category_name"
).drop(
    "product_category_name"
)

# Reordenando as colunas para colocar 'category_name' depois de 'product_id'
final_columns = [
    "product_id",
    "category_name",
] + [c for c in products_translated_df.columns if c not in ["product_id", "category_name"]]

products_reordered_df = products_translated_df.select(final_columns)

print("DataFrame após tradução e reordenação das colunas:")
products_reordered_df.show(5)

DataFrame após tradução e reordenação das colunas:
+--------------------+--------------+----------------+-----------------+-----------------+----------------+
|          product_id| category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+--------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|     perfumery|             225|               16|               10|              14|
|3aa071139cb16b67c...|           art|            1000|               30|               18|              20|
|96bd76ec8810374ed...|sports_leisure|             154|               18|                9|              15|
|cef67bcfe19066a93...|          baby|             371|               26|                4|              26|
|9dc1a7de274444849...|    housewares|             625|               20|               17|              13|
+--------------------+--------------+----------------+-----------------+-------------

In [9]:
# 5. Cálculo do Volume
# Multiplicando as dimensões para criar a coluna 'volume'
# e removendo as colunas originais
products_volume_df = products_reordered_df.withColumn(
    "volume_cm3",
    col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
).drop(
    "product_length_cm", "product_height_cm", "product_width_cm"
)

print("DataFrame final com a coluna de volume:")
products_volume_df.printSchema()

DataFrame final com a coluna de volume:
root
 |-- product_id: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- volume_cm3: double (nullable = true)



In [10]:
# 6. Exibição do Resultado Final
final_df = products_volume_df.select("product_id", "category_name", "product_weight_g", "volume_cm3")

print("Amostra do DataFrame final transformado:")
final_df.show()

Amostra do DataFrame final transformado:
+--------------------+--------------------+----------------+----------+
|          product_id|       category_name|product_weight_g|volume_cm3|
+--------------------+--------------------+----------------+----------+
|1e9e8ef04dbcff454...|           perfumery|             225|    2240.0|
|3aa071139cb16b67c...|                 art|            1000|   10800.0|
|96bd76ec8810374ed...|      sports_leisure|             154|    2430.0|
|cef67bcfe19066a93...|                baby|             371|    2704.0|
|9dc1a7de274444849...|          housewares|             625|    4420.0|
|41d3672d4792049fa...| musical_instruments|             200|    2090.0|
|732bd381ad09e530f...|          cool_stuff|           18350|   73920.0|
|2548af3e6e77a690c...|     furniture_decor|             900|   12800.0|
|37cc742be07708b53...|     home_appliances|             400|    5967.0|
|8c92109888e8cdf9d...|                toys|             600|    2040.0|
|14aa47b7fe5c25522...| 

In [12]:
final_df.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/products_cleaned_dataset.csv')

spark.stop()

26/04/03 22:51:36 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/03 22:51:37 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
